# MVP Análise de Dados e Boas Práticas

**Nome:** Matheus Henrique Santos da Silva

**Matrícula:**

Dataset: Online-Learning-Engagement-Dataset



# Descrição do Problema

Conjunto de Dados de Engajamento em Aprendizagem Online

Este conjunto de dados contém dados sintéticos que representam o engajamento e o comportamento de aprendizagem dos alunos em um ambiente de educação online. Inclui diversas características relacionadas a dados demográficos dos alunos, hábitos de estudo, atividade na plataforma e desempenho acadêmico.

O conjunto de dados foi projetado para simular cenários realistas de aprendizagem online e pode ser usado para mineração de dados educacionais, experimentos de aprendizado de máquina e projetos de análise.

As principais variáveis ​​incluem frequência de login dos alunos, horas de estudo, tempo de visualização de vídeos, envio de tarefas, tentativas de questionários e métricas de engajamento. Essas características ajudam a capturar como os alunos interagem com as plataformas de aprendizagem digital.

Este dataset será utilizado com o objetivo de mapear o perfil dos alunos, identificando a média de idade, sexo, a qual região pertence e a média de horas estudadas.



## Hipóteses do Problema

As hipóteses que tracei são as seguintes:

1. **A maioria dos alunos é do sexo feminino**

2. **A média de idade dos alunos está na casa dos 20 anos**

3. **A maioria dos alunos mora nos EUA**

## Tipo de Problema

Este é um problema de **regressão supervisionada**. Dado um conjunto de características (idade, sexo, numero de horas em estudo na semana), pode-se criar um modelo de regressão para prever as notas dos alunos.

## Seleção de Dados

Plataformas de educação online geram grandes quantidades de dados comportamentais. Compreender os padrões de engajamento dos alunos pode ajudar os educadores a aprimorar as experiências de aprendizagem e identificar alunos que podem estar em risco de evasão escolar. A base esta pronta para uso e foi recolhida no *Kaggle* no link: https://www.kaggle.com/datasets/ssssws/online-learning-engagement-dataset

## Atributos do Dataset

O dataset contém 50 mil linhas, com 18 colunas:

- ***student_id*** (Unique identifier assigned to each student in the dataset.)
- ***age*** (Age of the student participating in the online learning platform.)
- ***gender*** (Gender of the student (e.g., Male, Female).)
- ***country*** (Country where the student is located.)
- ***device_type*** (Type of device used to access the online learning platform (Laptop, Mobile, Tablet).)
- ***internet_speed_mbps*** (Internet connection speed of the student in megabits per second (Mbps).)
- ***study_hours_weekly*** (Average number of hours the student spends studying per week.)
- ***login_frequency_weekly*** (Number of times the student logs into the learning platform each week.)
- ***avg_session_duration_min*** (Average duration of a learning session in minutes.)
- ***video_watch_time_min*** (Total time spent watching course videos in minutes.)
- ***assignments_submitted***
- ***forum_posts***
- ***quiz_attempts***
- ***avg_quiz_score***
- ***attendance_rate***
- ***engagement_score***
- ***final_grade***
- ***dropout***

# Importação das Bibliotecas Necessárias e Carga de Dados

Esta seção consolida todas as importações de bibliotecas necessárias para a análise, visualização e pré-processamento dos dados, bem como o carregamento inicial do dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy.stats import f_oneway
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# @title
# carregamento do dataset
url = "https://raw.githubusercontent.com/Matheus-hss/Online-Learning-Engagement-Dataset/main/online_learning_engagement_dataset.csv"

df = pd.read_csv(url)

df.head()

# Análise de Dados

Nesta etapa de Análise de Dados Exploratória (EDA) sobre o dataset, vamos entender a distribuição, as relações e as características das variáveis, o que é crucial para as etapas subsequentes de pré-processamento e modelagem.

## Total e Tipo das Instâncias

O dataset possui 50 mil observações, com 18 colunas em formatos que variam de interger, float e object.

In [ ]:
# @title
print(f"Total de instâncias: {len(df)}")
print("\nTipos de dados por coluna:")
print(df.info())

In [ ]:
# @title
plt.figure(figsize=(7, 5))
# gráfico de barras simples
sns.countplot(x='country', data=df, hue= 'gender')
plt.title('Distribuição dos alunos por país')
plt.xlabel('País')
plt.ylabel('Contagem')
plt.show()

O gráfico de barras indica uma distribuição equilibrada de alunos por país, mas destaca que os EUA têm a maior concentração, predominando o gênero feminino.

## Estatísticas Descritivas

Estatísticas descritivas fornecem um resumo das características numéricas, incluindo média, desvio padrão, mínimo, máximo e quartis.

In [ ]:
# @title
# estatísticas descritivas básicas do dataset apenas das variaveis numéricas excluindo a coluna de id do aluno
df.drop(columns=['student_id']).select_dtypes(include=[np.number]).describe()

## Analise inicial

Há valores impossíveis em várias variáveis.

Exemplos:

* internet_speed_mbps: min = -7.3
* study_hours_weekly: min = -5.79
* avg_session_duration_min: min = -19.87
* video_watch_time_min: min = -237.96
* engagement_score: min < 0
* final_grade: min < 0

Isso indica erro de geração de dados ou falta de tratamento de outliers

## Distribuição

A distribuição de dados descreve como os valores de uma variável se espalham, ou seja, a frequência com que diferentes valores ocorrem. Entender a distribuição é crucial na análise de dados, pois revela padrões, tendências centrais, dispersão e a presença de valores atípicos (outliers). O histograma é uma ferramenta visual fundamental para representar essa distribuição, mostrando a forma dos dados, se são simétricos ou assimétricos, unimodais ou multimodais.

In [ ]:
#Função para construir gráfico de distribuição para cada uma das colunas do dataset
lista_colunas = df.drop(columns=['student_id', 'gender', 'country', 'device_type', 'dropout'])

for coluna in lista_colunas:
    plt.figure(figsize=(8, 6))
    sns.histplot(df[coluna], kde=True)
    plt.title(f'Distribuição de {coluna}')
    plt.xlabel(coluna)
    plt.show()


A maior parte das variaveis possui distribuição normal ou unfirme, em seguida vamos analisar com boxplot as possiveis variaveis com outliers

## Padronização e Boxplot

O gráfico boxplot nos permite comparar a média, mediana e desvio padrão de cada uma das variaveis com possiveis outliers indentificadas na analise inicial e saber se o tratamento é necessário.


In [ ]:
# @title
#Criando uma função para gerar um boxplot para cada uma das seguintes variaveis:
#'internet_speed_mbps',
#'study_hours_weekly',
#'avg_session_duration_min',
#'video_watch_time_min',
#'engagement_score',
#'final_grade'

box_analise = ['internet_speed_mbps',
    'study_hours_weekly',
    'avg_session_duration_min',
    'video_watch_time_min',
    'engagement_score',
    'final_grade']

scaler = StandardScaler() #usamos padronização para facilitar a leitura

df_scaled = pd.DataFrame(
    scaler.fit_transform(df[box_analise]),
    columns=box_analise
)

plt.figure(figsize=(12, 6))

sns.boxplot(data=df_scaled)

plt.title('Boxplot padronizado (z-score)')
plt.xticks(rotation=45)

plt.show()

Outliers — diagnóstico

🔻 Negativos: impossíveis do ponto de vista físico (ex: tempo negativo)

🔺 Positivos: distribuição com caudas mais pesadas que normal ou presença de ruído artificial extremo

Mesmo após padronização os outliers continuam extremos.

Isso indica que não são apenas escala → são valores realmente inconsistentes

video_watch_time_min e avg_session_duration_min
maior dispersão relativa e outliers mais extremos

study_hours_weekly
distribuição bem centrada menos extremos variável mais “estável”

engagement_score presença de valores negativos

final_grade caudas relevantes possível inconsistência de escala

Implicações para modelagem Sem tratamento → risco alto de:
* distorção de coeficientes (logit/OLS)
* splits ruins (árvores)
* overfitting em outliers

## Tratamento de Outliers

In [ ]:
#removendo valores impossiveis, exemplo 'study_hours_weekly < 0'

df[box_analise] = df[box_analise].clip(lower=0)

In [ ]:
#Tratar outliers extremos com winsorização

def winsorize(series, lower=0.01, upper=0.99):
    # series here is expected to be a pandas Series (single column)
    return series.clip(
        lower=series.quantile(lower),
        upper=series.quantile(upper)
    )

for col in box_analise:
    # Apply winsorize to each column individually
    df[col] = winsorize(df[col])

Rodar novamente o código do boxplot para verificar:

In [ ]:
box_analise = ['internet_speed_mbps',
    'study_hours_weekly',
    'avg_session_duration_min',
    'video_watch_time_min',
    'engagement_score',
    'final_grade']

scaler = StandardScaler() #usamos padronização para facilitar a leitura

df_scaled = pd.DataFrame(
    scaler.fit_transform(df[box_analise]),
    columns=box_analise
)

plt.figure(figsize=(12, 6))

sns.boxplot(data=df_scaled)

plt.title('Boxplot padronizado (z-score)')
plt.xticks(rotation=45)

plt.show()

Outliers ajustados.

## Tratamento de Valores Nulos

O dataset não possui valores nulos. No entanto, o tratamento de valores nulos é crucial e pode envolver imputação (preenchimento com média, mediana, moda) ou remoção de linhas/colunas.

In [ ]:
# Verificar a presença de valores nulos no dataset original
print("Valores nulos no dataset:")
df.isnull().sum()

## Matriz de Correlação

A matriz de correlação mede a força e a direção de uma relação linear que os atributos numéricos podem ter. Valores próximos a 1 indicam uma forte correlação positiva, -1 uma forte correlação negativa, e 0 ausência de correlação linear.

In [ ]:
# Matriz de correlação
print("\nMatriz de Correlação:")
corre = df.drop(columns=['student_id']).select_dtypes(include=[np.number]).iloc[:, :13].corr()
corre

In [ ]:
plt.figure(figsize=(8, 6))
# mapa de calor das variáveis numéricas
sns.heatmap(corre, annot=True, fmt=".2f")
plt.title('Matriz de Correlação das Características Numéricas do Dataset')
plt.show()

O mapa de calor da matriz de correlação revela fortes correlações positivas (acima de 0.7) entre as variaveis study_hours_weekly ↔ engagement_score → 0.79 e avg_quiz_score ↔ final_grade → 0.83.

Vamos remover a variavel engagement_score, manteremos avg_quiz_score pois ela possui poder explicativo em um modelo que busca prever a nota final dos alunos.

In [ ]:
#Removendo variaveis
df = df.drop(columns=['engagement_score'])

## Analise das Categóricas

Temos 3 variaveis categóricas em nosso dataset, vamos fazer uma breve analise antes de realizar o pré-processamento.

In [ ]:
#desbalanceamento

for col in ['gender', 'country', 'device_type']:
    print(df[col].value_counts(normalize = True))

As variaveis categóricas estão bem balanceadas e ja conseguimos traçar um perfil dos alunos. A maior parte é de mulheres na casa dos 30 anos que moram nos EUA e usam laptops para assistir as aulas.

In [ ]:
#Médias por grupo

for col in ['gender', 'country', 'device_type']:
    print(df.groupby(col)['final_grade'].mean())
#As médias por grupo em relação a média não divergem, para confirmar iremos rodar ANOVA

In [ ]:

groups = [group['final_grade'].values for _, group in df.groupby('country')]

f_oneway(*groups)

# aqui rodamos os testes separadamente para cada uma das três variaveis

Resultados dos testes anova

* device_type: F = 1.91/p-value = 0.147
  Não rejeitamos H₀ (α = 5%)

Não há evidência estatística de que o tipo de dispositivo afete a nota final

* gender: F = 2.05/p-value = 0.151
  Não rejeitamos H₀

Diferença de desempenho entre gêneros não é estatisticamente significativa

* country: F = 0.17/p-value = 0.973
  Fortíssima evidência de igualdade

Praticamente nenhuma diferença entre países


In [ ]:
#Vamos verificar se podem existir informações no nivel de interação entre as categóricas e as numéricas, exemplo: device_type vs estudo
df.groupby('device_type')[[
    'study_hours_weekly',
    'avg_session_duration_min',
    'avg_quiz_score'
]].mean()

Implicações para modelagem

Variáveis categóricas têm baixo poder explicativo não explicam variação de final_grade são, no máximo, variáveis secundárias, portanto serão removidas.



In [ ]:
#removendo variaveis categóricas
drop_cols = ['gender', 'country', 'device_type']
df = df.drop(columns=drop_cols)

# Pré-Processamento de Dados

O pré-processamento de dados é uma etapa crucial para preparar os dados para modelagem, garantindo que estejam no formato correto e otimizados para o desempenho do algoritmo.

In [ ]:
# Separar features (X) e target (y)
X = df.drop('final_grade', axis=1)
y = df['final_grade']

In [ ]:
# Dividir os dados em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
print(f"Dimensões de X_train: {X_train.shape}")
print(f"Dimensões de X_test: {X_test.shape}")
print(f"Dimensões de y_train: {y_train.shape}")
print(f"Dimensões de y_test: {y_test.shape}")

## Normalização

A normalização escala os dados para um intervalo fixo, geralmente entre 0 e 1. É útil quando o algoritmo de machine learning assume que as características estão em uma escala semelhante.



In [ ]:
# Inicializar o MinMaxScaler
scaler_norm = MinMaxScaler()

In [ ]:
# Aprende min e max APENAS de X_train
scaler_norm.fit(X_train)
X_train_normalized = scaler_norm.transform(X_train)
# Usa a média e o desvio padrão aprendidos de X_train
X_test_normalized = scaler_norm.transform(X_test)

In [ ]:
# Exibir as primeiras linhas dos dados normalizados (como DataFrame para melhor visualização)
df_normalized = pd.DataFrame(X_train_normalized, columns=X_train.columns)

In [ ]:
print("\nPrimeiras 5 linhas dos dados normalizados (treino):")
print(df_normalized.head())

# Respondendo nossas hipóteses



## Hipótese 1

hipótese 1:
**A maioria dos alunos é do sexo feminino**

R: Sim conforme vimos no gráfico de colunas e depois quando averiguamos o
balancemanto das variaveis categóricas vemos que o sexo feminino é majoritario
por pouco.

* Feminino: 50,3%
* Masculino: 49,7%

## Hipótese 2

hipótese 2: **A média de idade dos alunos está na casa dos 20 anos**

R:Não, a média de idade dos alunos conforme as estatísticas descritivas básicas do dataset está em 33 anos.

## Hipótese 3

hipótese 3: **A maioria dos alunos mora nos EUA**

R:Não, pelo o que vimos no dataset quando analisamos as variaveis categóricas não podemos fazer confirmar que a maioria dos estudantes são dos EUA, pois os 3 primeiros países possuem uma distribuição de alunos semelhantes:

* USA 16,8%
* India 16,7%
* Canada 16,7%

# Conclusão

A análise e pré-processamento do dataset demonstram a importância de entender a estrutura dos dados antes da modelagem. O dataset é limpo e balanceado, com características numéricas bem definidas que permitem uma clara separação entre as variaveis numéricas e categóricas. A análise exploratória revelou correlações importantes entre as características e a eficácia de visualizações como boxplots para distinguir as classes nos apresentou um padrão claro de estudantes da base. Os próximos passos importantes seriam as estapas de treino e teste na modelagem seguido pela analise das métricas do modelo.